# Entity Swap Eval on Colab

This notebook is meant to run from VS Code with the Google Colab kernel selected. Cells execute on the Colab VM, not on the local machine.

Do not upload or commit your local `generated_graphs/` directory. It contains the large raw graph files and is intentionally ignored by Git.

The eval needs `generated_graphs/000.sng.pt` through `099.sng.pt` and `bats_analogies.txt`. This repo tracks the compact labeled summaries in `labeled_summary/entmax/alpha_0.50/node_0.02`, so the setup below creates a fresh small `generated_graphs/` directory inside Colab from those tracked files. If you already renamed those files to `000.sng.pt` through `099.sng.pt`, the prep cell can use those names too.

In [1]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
REPO_URL = "https://github.com/IamKrill1n/circuit_tracer_mod.git"
REPO_REF = "c0c035a"
REPO_DIR = "/content/circuit_tracer_mod"

In [ ]:
# from getpass import getpass
# from pathlib import Path
# import os
# import shutil
# import subprocess

# repo_dir = Path(REPO_DIR)

# env = os.environ.copy()
# env["GIT_TERMINAL_PROMPT"] = "0"
# token = env.get("GITHUB_TOKEN") or getpass(
#     "GitHub token with repo read access; leave blank only if the repo is public: "
# ).strip()
# if token:
#     askpass = Path("/tmp/git-askpass.sh")
#     askpass.write_text(
#         "#!/bin/sh\n"
#         "case \"$1\" in\n"
#         "  *Username*) echo x-access-token ;;\n"
#         "  *Password*) echo \"$GITHUB_TOKEN\" ;;\n"
#         "esac\n",
#         encoding="utf-8",
#     )
#     askpass.chmod(0o700)
#     env["GIT_ASKPASS"] = str(askpass)
#     env["GITHUB_TOKEN"] = token

# os.chdir("/content")
# if repo_dir.exists() and not (repo_dir / ".git").exists():
#     shutil.rmtree(repo_dir)
# if not (repo_dir / ".git").exists():
#     subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True, env=env)

# subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "--tags"], check=True, env=env)
# subprocess.run(["git", "-C", REPO_DIR, "checkout", REPO_REF], check=True, env=env)
# os.chdir(REPO_DIR)
# print(f"checked out {REPO_REF} in {Path.cwd()}")

CalledProcessError: Command '['git', 'clone', 'https://github.com/IamKrill1n/circuit_tracer_mod.git', '/content/circuit_tracer_mod']' returned non-zero exit status 128.

If you do not want to use a GitHub token, upload the local archive `/tmp/circuit_tracer_mod_colab_source.tgz` to Colab as `/content/circuit_tracer_mod_colab_source.tgz`, then run the fallback cell below instead of the clone cell.

In [4]:
# Fallback source setup. Run only if the GitHub clone cell failed.
%cd /content
!rm -rf {REPO_DIR}
!mkdir -p {REPO_DIR}
!tar -xzf /content/circuit_tracer_mod_colab_source.tgz -C {REPO_DIR}
%cd {REPO_DIR}

/content
tar (child): /content/circuit_tracer_mod_colab_source.tgz: Cannot open: No such file or directory
tar (child): Error is not recoverable: exiting now
tar: Child returned status 2
tar: Error is not recoverable: exiting now
/content/circuit_tracer_mod


In [ ]:
!python -m pip install -U pip
!python -m pip install -e ".[dev]"

In [ ]:
from pathlib import Path
import shutil

repo_root = Path.cwd()
summary_dir = Path("labeled_summary/entmax/alpha_0.50/node_0.02")
graph_dir = Path("generated_graphs")

print(f"cwd: {repo_root}")
assert Path("pyproject.toml").exists(), (
    "run the clone or fallback setup cell first, then %cd /content/circuit_tracer_mod"
)

summary_sources = sorted(summary_dir.glob("*_labeled_summary_graph.pt"))
renamed_sources = sorted(summary_dir.glob("*.sng.pt"))
if len(summary_sources) == 100:
    sources = [(source, source.name.removesuffix("_labeled_summary_graph.pt")) for source in summary_sources]
elif [path.name for path in renamed_sources] == [f"{i:03d}.sng.pt" for i in range(100)]:
    sources = [(source, source.name.removesuffix(".sng.pt")) for source in renamed_sources]
else:
    raise AssertionError(
        f"expected 100 compact summaries or 100 renamed .sng.pt files in {summary_dir}; "
        f"found {len(summary_sources)} compact summaries and {len(renamed_sources)} .sng.pt files"
    )

if graph_dir.exists():
    shutil.rmtree(graph_dir)
graph_dir.mkdir()

for source, stem in sources:
    shutil.copy2(source, graph_dir / f"{stem}.sng.pt")

paths = sorted(graph_dir.glob("*.sng.pt"))
expected = [f"{i:03d}.sng.pt" for i in range(100)]
assert [p.name for p in paths] == expected
assert len(Path("bats_analogies.txt").read_text(encoding="utf-8").splitlines()) == 100
total_mb = sum(path.stat().st_size for path in paths) / 1024 / 1024
print(f"prepared {len(paths)} Colab input graphs in {graph_dir} ({total_mb:.1f} MB)")

If Hugging Face blocks model or transcoder downloads, run the login cell below and use a token that can access `google/gemma-2-2b`.

In [ ]:
# Optional: uncomment if model download requires authentication.
# from huggingface_hub import notebook_login
# notebook_login()

In [ ]:
!mkdir -p runs eval_outputs/entity_swap_colab_smoke
!python -u eval/eval_entity_swap.py \
  --graph-dir generated_graphs \
  --analogies-file bats_analogies.txt \
  --relations 0 \
  --negation-coefficients -2 \
  --addition-coefficients 2 \
  --output-dir eval_outputs/entity_swap_colab_smoke \
  --device cuda \
  --dtype bfloat16 \
  2>&1 | tee runs/entity_swap_smoke.log

In [ ]:
!ls -lh eval_outputs/entity_swap_colab_smoke
!head -5 eval_outputs/entity_swap_colab_smoke/swap_summary.csv

In [ ]:
!mkdir -p runs eval_outputs/entity_swap_colab_sample10
!nohup python -u eval/eval_entity_swap.py \
  --graph-dir generated_graphs \
  --analogies-file bats_analogies.txt \
  --sample-pairs-per-relation 10 \
  --random-state 42 \
  --output-dir eval_outputs/entity_swap_colab_sample10 \
  --device cuda \
  --dtype bfloat16 \
  > runs/entity_swap_sample10.log 2>&1 & echo $! > runs/entity_swap_sample10.pid
!cat runs/entity_swap_sample10.pid

In [ ]:
# !mkdir -p runs eval_outputs/entity_swap_colab
# !nohup python -u eval/eval_entity_swap.py \
#   --graph-dir generated_graphs \
#   --analogies-file bats_analogies.txt \
#   --output-dir eval_outputs/entity_swap_colab \
#   --device cuda \
#   --dtype bfloat16 \
#   > runs/entity_swap_full.log 2>&1 & echo $! > runs/entity_swap_full.pid
# !cat runs/entity_swap_full.pid

In [ ]:
!tail -n 80 runs/entity_swap_full.log
!ls -lh eval_outputs/entity_swap_colab || true

In [ ]:
# Persist outputs before the Colab runtime is recycled.
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/circuit_tracer_entity_swap
# !cp -r eval_outputs/entity_swap_colab runs/entity_swap_full.log /content/drive/MyDrive/circuit_tracer_entity_swap/